In [7]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import (
    BertTokenizer, BertForTokenClassification, BertForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorForTokenClassification
)
import warnings
warnings.filterwarnings('ignore')

# ======================== 1. 数据加载与预处理 ========================
# 加载数据集
df = pd.read_csv('D:/Doc/biYeSheJi/cloud/src/trainset/texttrain_backward.csv', encoding='utf-8')

# 定义标签映射
slot_labels = ['o', 'b', 'i']
slot2id = {label: idx for idx, label in enumerate(slot_labels)}
id2slot = {idx: label for idx, label in enumerate(slot_labels)}

# 意图标签：从训练集自动提取
intent_labels = sorted(df['intent'].unique().tolist())
intent2id = {label: idx for idx, label in enumerate(intent_labels)}
id2intent = {idx: label for idx, label in enumerate(intent_labels)}
# 关键：判断是否单意图，后续适配损失函数
is_single_intent = len(intent_labels) == 1

# 初始化tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# ---------------------- 槽位填充数据预处理 ----------------------
def tokenize_slot_data(examples):
    tokenized_inputs = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=10
    )

    labels = []

    for i, slots in enumerate(examples['slots']):
        # tokenizer 后的 token 数量（不含 special token）
        tokens = tokenizer.tokenize(examples['text'][i])
        
        if len(slots) != len(tokens):
            print(f"警告：第{i}条数据 token 数量({len(tokens)})与标签数量({len(slots)})不一致，跳过该条")
            labels.append([-100] * 10)
            continue

        # 加上 special token 的 -100
        label_ids = []

        # CLS
        label_ids.append(-100)

        # token 对应标签
        for slot in slots:
            label_ids.append(slot2id.get(slot, 0))

        # SEP
        label_ids.append(-100)

        # padding 到 max_length
        while len(label_ids) < 10:
            label_ids.append(-100)

        # 截断
        label_ids = label_ids[:10]

        labels.append(label_ids)

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

# ---------------------- 意图分类数据预处理 ----------------------
def tokenize_intent_data(examples):
    tokenized_inputs = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=30
    )
    # 关键：单意图时，标签转为Float型（适配回归损失）；多意图时保持Long型
    if is_single_intent:
        tokenized_inputs['labels'] = torch.tensor([intent2id[intent] for intent in examples['intent']], dtype=torch.float32)
    else:
        tokenized_inputs['labels'] = [intent2id[intent] for intent in examples['intent']]
    return tokenized_inputs

# 转换数据集
from datasets import Dataset
dataset = Dataset.from_pandas(df)

# 1. 槽位任务数据集
slot_dataset = dataset.map(
    tokenize_slot_data,
    batched=True,
    remove_columns=dataset.column_names
)
slot_dataset = slot_dataset.train_test_split(test_size=0.2, seed=42)

# 2. 意图任务数据集
intent_dataset = dataset.map(
    tokenize_intent_data,
    batched=True,
    remove_columns=dataset.column_names
)
intent_dataset = intent_dataset.train_test_split(test_size=0.2, seed=42)

# ======================== 2. 自定义意图训练器（适配单意图） ========================
class IntentTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # 单意图时用MSELoss（回归），多意图时用默认CrossEntropyLoss
        self.loss_fn = nn.MSELoss() if is_single_intent else None

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels").to(device)
        outputs = model(**inputs)
        logits = outputs.logits
        
        if is_single_intent:
            # 单意图：回归损失（logits维度[batch_size, 1]，转为float）
            loss = self.loss_fn(logits.squeeze().float(), labels.float())
        else:
            # 多意图：默认分类损失
            loss = nn.CrossEntropyLoss()(logits, labels.long())
        
        return (loss, outputs) if return_outputs else loss

# ======================== 3. 模型训练 ========================
# ---------------------- 1. 训练槽位填充模型 ----------------------
print("\n===== 开始训练槽位填充模型 =====")
slot_model = BertForTokenClassification.from_pretrained(
    'bert-base-chinese',
    num_labels=len(slot_labels),
    ignore_mismatched_sizes=True
).to(device)

# 槽位训练参数（移除deprecated的logging_dir）
slot_training_args = TrainingArguments(
    output_dir='./slot_results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,  # 移除logging_dir，使用默认日志路径
    eval_strategy='epoch',
    save_strategy='epoch',
    metric_for_best_model='loss',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

# 槽位训练器
slot_trainer = Trainer(
    model=slot_model,
    args=slot_training_args,
    train_dataset=slot_dataset['train'],
    eval_dataset=slot_dataset['test'],
    data_collator=DataCollatorForTokenClassification(tokenizer, padding=True)
)
slot_trainer.train()

# ---------------------- 2. 训练意图分类模型 ----------------------
print("\n===== 开始训练意图分类模型 =====")
intent_model = BertForSequenceClassification.from_pretrained(
    'bert-base-chinese',
    num_labels=len(intent_labels),
    ignore_mismatched_sizes=True
).to(device)

# 意图训练参数
intent_training_args = TrainingArguments(
    output_dir='./intent_results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    metric_for_best_model='loss',
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

# 意图训练器（使用自定义IntentTrainer适配单意图）
intent_trainer = IntentTrainer(
    model=intent_model,
    args=intent_training_args,
    train_dataset=intent_dataset['train'],
    eval_dataset=intent_dataset['test']
)
intent_trainer.train()

# ======================== 4. 推理函数（合并槽位+意图） ========================
def predict(text):
    # 预处理输入
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding='max_length',
        max_length=30,
        truncation=True
    ).to(device)
    
    # 关闭梯度计算
    with torch.no_grad():
        # 1. 槽位预测
        slot_logits = slot_model(**inputs).logits
        slot_preds = torch.argmax(slot_logits, dim=-1).squeeze().cpu().numpy()
        
        # 2. 意图预测（适配单意图）
        intent_logits = intent_model(**inputs).logits
        if is_single_intent:
            # 单意图直接返回唯一标签
            intent_label = intent_labels[0]
        else:
            intent_pred = torch.argmax(intent_logits, dim=-1).item()
            intent_label = id2intent[intent_pred]
    
    # 解析槽位（过滤特殊token）
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    slot_results = []
    for token, pred in zip(tokens, slot_preds):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        slot_label = id2slot[pred] if pred != -100 else 'o'
        slot_results.append((token, slot_label))
    
    return {
        '输入文本': text,
        '预测意图': intent_label,
        '槽位预测': slot_results
    }


使用设备: cuda


Map:   0%|          | 0/65 [00:00<?, ? examples/s]

警告：第60条数据 token 数量(4)与标签数量(5)不一致，跳过该条


Map:   0%|          | 0/65 [00:00<?, ? examples/s]


===== 开始训练槽位填充模型 =====


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [10]:
# 测试推理
test_text = "不要前进了，往后退2米。"
result = predict(test_text)
print("\n===== 推理结果 =====")
print(f"输入文本：{result['输入文本']}")
print(f"预测意图：{result['预测意图']}")
print(f"槽位预测：{result['槽位预测']}")


===== 推理结果 =====
输入文本：不要前进了，往后退2米。
预测意图：B
槽位预测：[('不', 'o'), ('要', 'o'), ('前', 'o'), ('进', 'o'), ('了', 'o'), ('，', 'o'), ('往', 'o'), ('后', 'o'), ('退', 'o'), ('2', 'b'), ('米', 'o'), ('。', 'o')]


In [ ]:
import pandas as pd

def normalize_slots(slot_str):
    result = []
    found_b = False
    in_span = False

    for ch in slot_str:
        if ch == 'o':
            result.append('o')
            in_span = False
        else:  # 非o
            if not found_b:
                result.append('b')
                found_b = True
                in_span = True
            elif in_span:
                result.append('i')
            else:
                result.append('o')

    return ''.join(result)

df = pd.read_csv('D:/Doc/biYeSheJi/cloud/src/trainset/texttrain_new_mod.csv', encoding='utf-8')
df_mod = pd.DataFrame(columns=['text', 'intent', 'slots'])
for index, row in df.iterrows():
    text = row['text']
    res = predict(text)
    intent = res['预测意图']
    #将slot的结果提取为字符串
    slot_str = ''.join([f"{label}" for token, label in res['槽位预测']])
    df.loc[index,'intent'] = intent
    df.loc[index,'slots'] = slot_str
df["slots"]  = df["slots"].apply(normalize_slots)
df.to_csv('D:/Doc/biYeSheJi/cloud/src/trainset/texttrain_new_mod.csv', index=False, encoding='utf-8')